# Pardes semantic model on Colab (free T4 works)

Benchmarks the Dicta model, fine-tunes it on the library's links, distills it into a fast 7-layer / 384-dim model,
measures each, and embeds the whole library with the chosen one. Everything that takes long is saved in Google Drive
(`WORK` below) and **resumes where it stopped**: when the session ends, reconnect, run the setup cells (1-4) again,
then the cell that was running.

Runtime → Change runtime type → **T4 GPU**. Rough times on a free T4 (they vary): library 10 min, baseline 15 min,
fine-tuning 4-6 h (over two sessions if need be), distillation 1-2 h, full library with the student 2-3 h.

## 1. GPU and Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/pardes-semantic'   # what must survive the session
LOCAL = '/content/work'                           # the 8 GB library: on the session's disk, fetched again if lost
import os; os.makedirs(WORK, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
os.environ['WORK'], os.environ['LOCAL'] = WORK, LOCAL

## 2. The code

The repository is public: the notebook clones its `main` branch.

In [ ]:
!rm -rf /content/repo && git clone -q --depth 1 https://github.com/efrul/Pardes /content/repo
%cd /content/repo/semantic-model
!git log --oneline -1

## 3. Packages (Colab's own CUDA torch is kept)

In [ ]:
!pip -q install "transformers==4.57.6" "tokenizers>=0.22,<0.23" "sentence-transformers==5.1.2" "datasets>=3,<5" "accelerate>=1.0" "onnx>=1.19" "onnxscript>=0.5" "onnxruntime==1.31.0" "zstandard>=0.23"
import torch; print(torch.__version__, torch.cuda.get_device_name(0), 'bf16' if torch.cuda.is_bf16_supported() else 'fp16')

## 4. The library, the benchmark and the training pairs

In [ ]:
!test -f $LOCAL/library/seforim.db || python fetch_library.py --out $LOCAL/library
!test -f $WORK/eval/queries.jsonl || python build_eval_set.py --db $LOCAL/library/seforim.db --out $WORK/eval
!test -f $WORK/train/pairs.jsonl || python build_train_pairs.py --db $LOCAL/library/seforim.db --out $WORK/train/pairs.jsonl

## 5. Measure a model

`measure(name, source)`: exports it to the app's int8 ONNX model, embeds the benchmark corpus on the GPU (as the
library would be), and scores it with the app's int8 query model. `source=None` is the Dicta model as is.

In [ ]:
import subprocess, pathlib
def sh(cmd):
    print('$', cmd); subprocess.run(cmd, shell=True, check=True)
def measure(name, source=None):
    src = f'--source {source}' if source else ''
    model, vec, rep = f'{WORK}/model-{name}', f'{WORK}/eval/vec-{name}', f'{WORK}/eval/report-{name}'
    if not pathlib.Path(model, 'pardes-model.json').exists():
        sh(f'python export_onnx.py {src} --out {model}')
    if not pathlib.Path(vec, 'meta.txt').exists():
        sh(f'python embed_corpus.py --corpus {WORK}/eval/corpus.jsonl --backend torch --device cuda --dtype float16 {src} --out {vec}')
    sh(f'python evaluate.py --eval {WORK}/eval --vectors {vec} --model {model} --out {rep}')
measure('base')

## 6. Fine-tune (resumes from its last checkpoint)

In [ ]:
# 150k pairs and a batch of 128 suit a T4 (16 GB); on an A100 drop --limit and --batch-size
sh(f'python train_model.py --pairs {WORK}/train/pairs.jsonl --out {WORK}/teacher --limit 150000 --batch-size 128 --mini-batch 16')
measure('teacher', f'{WORK}/teacher')

## 7. Distill into the fast model (7 layers, 384 dimensions)

In [ ]:
sh(f'python distill_model.py --teacher {WORK}/teacher --pairs {WORK}/train/pairs.jsonl --out {WORK}/student --texts 300000')
measure('student', f'{WORK}/student')

## 8. Compare

In [ ]:
for name in ('base', 'teacher', 'student'):
    p = pathlib.Path(WORK, 'eval', f'report-{name}', 'report.md')
    if p.exists(): print(f'===== {name}'); print(p.read_text())

## 9. The whole library with the chosen model (resumable)

Then download `WORK/vectors` and `WORK/model-<chosen>` from Drive: the app's vector index is built from them
(`./gradlew :SeforimApp:buildVectorIndex`, see README.md).

In [ ]:
CHOSEN = 'student'   # or 'teacher' / None for the Dicta model (much slower over 3.5M lines)
src = f'--source {WORK}/{CHOSEN}' if CHOSEN else ''
sh(f'python embed_corpus.py --db {LOCAL}/library/seforim.db --backend torch --device cuda --dtype float16 {src} --out {WORK}/vectors')